In [1]:
# Load data and split
import pandas as pd
import numpy as np
import joblib
import os

df_model = pd.read_csv("../data/processed/df_young_model_ready_2024.csv")

y = df_model["high_risk"].astype(int)
A = df_model[["sex_of_driver", "age_band_of_driver"]].copy()
X = df_model.drop(columns=["high_risk", "sex_of_driver", "age_band_of_driver"])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, A, test_size=0.2, stratify=y, random_state=42
)

In [2]:
# Train all three models
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

lr_bal = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))
])
lr_bal.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

xgb_clf = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric="logloss"
)
xgb_clf.fit(X_train, y_train)

print("All models trained!")

All models trained!


In [3]:
# Save models
os.makedirs('../models', exist_ok=True)

joblib.dump(lr_bal, '../models/lr_balanced.pkl')
joblib.dump(rf, '../models/rf_balanced.pkl')
joblib.dump(xgb_clf, '../models/xgb.pkl')
joblib.dump(X_train.columns.tolist(), '../models/feature_names.pkl')

print("Models saved successfully!")

Models saved successfully!
